# LEGA-C DR3 against DR2

- Cross-match, spectra, selection and paired fits, DR3 vs DR2.

In [1]:
from pathlib import Path
import json
import sys
import h5py
import numpy as np
import pandas as pd
import astropy.units as u
from astropy.coordinates import SkyCoord
from astropy.io import fits
from astropy.table import Table
from numpy.polynomial import legendre
from specutils.utils.wcs_utils import air_to_vac

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'scripts/experiment.py').exists())
sys.path.insert(0, str(ROOT))
from scripts.experiment import validate_result
from ceridwen.observation.absorption_features import ABSORPTION_FEATURES, feature_windows

OUT = ROOT / 'results/legac-dr3-2026-11-11'
RAW = ROOT / 'data/raw'
TEST_SET = [t['spect_id'] for t in json.loads((ROOT / 'results/quiescent-test-set/targets.json').read_text())['targets']]
DR2_135 = [t['spect_id'] for t in json.loads((ROOT / 'results/dr2-refit-2026-10-12/targets.json').read_text())['targets']]
DR2_187 = [t['spect_id'] for t in json.loads((ROOT / 'results/dr2-quiescent-new-defaults/targets.json').read_text())['targets']]
WINDOW = (4074.5, 4666.1)  # chronometer arm (i) rest_range [vacuum A]
TELLURIC_AIR = (7590.0, 7660.0)  # notebook telluric_air mask [observed air A]
# DR2 telluric-corrected ranges [observed air A], Straatman et al. 2018 Sect. 2 step 5
DR2_TELLURIC = [(6865, 6960), (7160, 7330), (7580, 7710), (8070, 8400), (8900, 9200)]
assert len(TEST_SET) == 10 and len(DR2_135) == 135 and len(DR2_187) == 187

- Catalogue rows by `SPECT_ID`; DR3 renames `z`, `_err` to `Z`, `_ERR`.

In [2]:
def text(value):
    return (value.decode() if isinstance(value, bytes) else str(value)).strip()


def catalogue(release):
    path = RAW / 'legac_dr2/legaCdr2.fits.gz' if release == 'dr2' else RAW / 'legac_dr3/legac_DR3.fits'
    table = Table.read(path, hdu=1).to_pandas()
    if release == 'dr2':
        table = table.rename(columns=lambda c: 'Z' if c == 'z' else c.removesuffix('_err') + '_ERR' if c.endswith('_err') else c)
    for column in ('SPECT_ID', 'Filename'):
        table[column] = table[column].map(text)
    return table


def spectrum(release, filename):
    with fits.open(RAW / f'legac_{release}/sp' / filename) as hdulist:
        data = hdulist[1].data
        return {k: np.asarray(data[k][0], float) for k in ('WAVE', 'FLUX', 'ERR', 'QUAL')}, dict(hdulist[0].header)


CAT = {release: catalogue(release) for release in ('dr2', 'dr3')}
BY_ID = {release: table.set_index('SPECT_ID') for release, table in CAT.items()}
{release: len(table) for release, table in CAT.items()}

{'dr2': 1988, 'dr3': 4081}

- DR2 135 and test set in DR3: same `OBJECT`, separation, Δz.

In [3]:
def crossmatch(ids):
    dr2, dr3 = BY_ID['dr2'].loc[ids], BY_ID['dr3'].reindex(ids)
    sep = SkyCoord(dr2.RAJ2000.to_numpy() * u.deg, dr2.DECJ2000.to_numpy() * u.deg).separation(
        SkyCoord(dr3.RAJ2000.to_numpy() * u.deg, dr3.DECJ2000.to_numpy() * u.deg)).arcsec
    return pd.DataFrame(dict(target=ids, in_dr3=dr3.OBJECT.notna().to_numpy(), same_object=(dr2.OBJECT == dr3.OBJECT).to_numpy(),
                             separation_arcsec=sep, z_dr2=dr2.Z.to_numpy(), z_dr3=dr3.Z.to_numpy(), dz=(dr3.Z - dr2.Z).to_numpy()))


match = pd.concat([crossmatch(DR2_135).assign(sample='dr2_135'), crossmatch(TEST_SET).assign(sample='test_set')])
match.to_csv(OUT / 'crossmatch.csv', index=False)
match.groupby('sample').agg(galaxies=('target', 'size'), in_dr3=('in_dr3', 'sum'), same_object=('same_object', 'sum'),
                            max_separation_arcsec=('separation_arcsec', 'max'), max_abs_dz=('dz', lambda d: d.abs().max()))

,galaxies,in_dr3,same_object,max_separation_arcsec,max_abs_dz
sample,,,,,
dr2_135,135,135,135,0.0,0.002900
test_set,10,10,10,0.0,0.000528


- Test-set spectra: grid, good pixels, flux and error ratios.
- Good pixel: `QUAL` 0, `ERR` > 0, finite flux, outside the A band.
- Smooth flux ratio: order-5 Legendre fit to DR3/DR2.
- `chi`: DR3 minus scaled DR2, over the combined error.

In [4]:
def good(s):
    return (s['QUAL'] == 0) & (s['ERR'] > 0) & np.isfinite(s['FLUX'])


def compare(target):
    (s2, h2), (s3, h3) = (spectrum(r, BY_ID[r].loc[target, 'Filename']) for r in ('dr2', 'dr3'))
    w = s2['WAVE']
    g2, g3 = good(s2), good(s3)
    both = g2 & g3 & ~((w >= TELLURIC_AIR[0]) & (w <= TELLURIC_AIR[1]))
    ratio = s3['FLUX'][both] / s2['FLUX'][both]
    wb = w[both]
    x = 2 * (wb - wb.min()) / (wb.max() - wb.min()) - 1
    coef = legendre.legfit(x, ratio, 5)
    curve = legendre.legval(np.linspace(-1, 1, 200), coef)
    at = legendre.legval(x, coef)
    chi = (s3['FLUX'][both] - at * s2['FLUX'][both]) / np.hypot(s3['ERR'][both], at * s2['ERR'][both])
    in_tell = np.any([(wb >= lo) & (wb <= hi) for lo, hi in DR2_TELLURIC], axis=0)
    err = s3['ERR'][both] / s2['ERR'][both]
    c2, c3 = BY_ID['dr2'].loc[target], BY_ID['dr3'].loc[target]
    return dict(
        target=target, grid_identical=bool(np.array_equal(s2['WAVE'], s3['WAVE'])),
        wavelmin_dr2=h2['WAVELMIN'] * 10, wavelmin_dr3=h3['WAVELMIN'] * 10,
        wavelmax_dr2=h2['WAVELMAX'] * 10, wavelmax_dr3=h3['WAVELMAX'] * 10,
        good_dr2=int(g2.sum()), good_dr3=int(g3.sum()), good_only_dr2=int((g2 & ~g3).sum()), good_only_dr3=int((~g2 & g3).sum()),
        flux_ratio_median=np.median(ratio), flux_ratio_blue=curve[0], flux_ratio_red=curve[-1],
        flux_ratio_peak_to_peak=(curve.max() - curve.min()) / np.median(ratio),
        chi_std=np.std(chi), chi_std_dr2_telluric=np.std(chi[in_tell]), chi_std_outside=np.std(chi[~in_tell]),
        err_ratio_median=np.median(err), err_ratio_over_flux_ratio=np.median(err / ratio),
        sn_ratio_median=np.median(ratio / err),
        sn_cat_dr2=c2.SN, sn_cat_dr3=c3.SN,
        sigma_star_dr2=c2.SIGMA_STARS_PRIME, sigma_star_dr3=c3.SIGMA_STARS_PRIME,
        sigma_star_err_dr2=c2.SIGMA_STARS_PRIME_ERR, sigma_star_err_dr3=c3.SIGMA_STARS_PRIME_ERR,
        flag_spec_dr3=int(c3.FLAG_SPEC), flag_morph_dr3=int(c3.FLAG_MORPH))


spectra = pd.DataFrame([compare(t) for t in TEST_SET])
spectra.to_csv(OUT / 'spectra-dr3-vs-dr2.csv', index=False)
spectra.round(3).set_index('target').T

target,M1_210210,M8_150848,M5_238314,M3_120308,M2_134021,M7_146213,M3_117010,M4_107370,M14_37219,M8_161113
grid_identical,True,True,True,True,True,True,True,True,True,True
wavelmin_dr2,6186.1,6740.5,6365.5,6721.9,6677.5,6585.1,5974.9,6830.5,6192.7,6239.5
wavelmin_dr3,6189.7,6744.1,6369.1,6725.5,6680.5,6587.5,5978.5,6833.5,6195.7,6242.5
wavelmax_dr2,8900.5,9229.9,8828.5,9203.5,9163.9,9066.1,8410.9,9324.1,8642.5,8694.1
wavelmax_dr3,8631.7,9225.1,8824.9,9198.7,9159.1,9062.5,8407.3,9320.5,8638.3,8689.3
good_dr2,4053,4099,4061,4064,4097,4074,3989,4056,4041,4042
good_dr3,4053,4119,4052,4045,3915,4056,3985,4076,4056,4044
good_only_dr2,13,10,31,38,197,54,26,27,7,21
good_only_dr3,13,30,22,19,15,36,22,47,22,23
flux_ratio_median,0.86,0.941,1.086,0.871,0.828,1.021,0.881,0.864,0.889,1.123


- DR2 cuts of the notebook, and their DR3 analogue.
- DR3 flags: `FLAG_PRIMARY` 1, `FLAG_SPEC` 0, `FLAG_MORPH` 0 (van der Wel et al. 2021 Sect. 4.2).
- DR2 `f_use` = primary, `f_spec`, `f_z`, `f_ppxf` all good (Straatman et al. 2018 Sect. 3.3).
- DR3 emission EWs are positive, DR2 negative: sign check below.

In [5]:
shared = BY_ID['dr2'].index.intersection(BY_ID['dr3'].index)
ew2, ew3 = BY_ID['dr2'].loc[shared, 'OII_3727_EW'], BY_ID['dr3'].loc[shared, 'OII_3727_EW']
ok = ew2.notna() & ew3.notna() & (ew2.abs() > 1)
assert np.median(ew3[ok] / ew2[ok]) < -0.9
PHOT_COLUMNS = ['Area', 'Sat', 'Cfl', 'Flag', 'NUVMag', 'RMag', 'JMag']


def parent(release):
    phot = Table.read(RAW / f'cosmos2015/cosmos2015_legac_{release}_photometry_1arcsec.fits').to_pandas().set_index('LEGAC_INDEX')
    return CAT[release].join(phot[PHOT_COLUMNS], how='inner')


def select(p, release):
    counts = {'catalogue x COSMOS2015': len(p)}
    if release == 'dr2':
        q = (p.f_use == 1) & (p.f_ppxf == 0) & (p.f_z == 0) & (p.f_int == 0)
    else:
        q = (p.FLAG_PRIMARY == 1) & (p.FLAG_SPEC == 0) & p.Z.notna() & p.SIGMA_STARS_PRIME.notna() & (p.FLAG_MORPH == 0)
    q &= (p.SN > 0) & (p.Z >= 0.6) & (p.Z < 1.0) & (p[['NUVMag', 'RMag', 'JMag']] > -40).all(axis=1)
    p = p[q]; counts['quality flags, z, S/N, NUV r J'] = len(p)
    nuv_r, r_j = p.NUVMag - p.RMag, p.RMag - p.JMag
    p = p[(nuv_r > 3 * r_j + 1) & (nuv_r > 3.1)]; counts['NUVrJ quiescent'] = len(p)
    ew = p.OII_3727_EW
    p = p[((ew > -5) if release == 'dr2' else (ew < 5)) | ew.isna()]; counts['[O II] EW below 5 A'] = len(p)
    detected = lambda line: (p[f'{line}_EW'] / p[f'{line}_EW_ERR']).abs() >= 3
    p = p[~(detected('OII_3727') | detected('OIII_5007'))]; counts['no 3 sigma [O II], [O III]'] = len(p)
    p = p[(p.Area == 0) & (p.Sat == 0) & (p.Cfl == 1) & (p.Flag == 0)]; counts['clean COSMOS2015 photometry'] = len(p)
    p = p.sort_values(['SN', 'SPECT_ID'], ascending=[False, True]).drop_duplicates('OBJECT', keep='first')
    counts['one spectrum per OBJECT'] = len(p)
    return p, counts


SELECTED, COUNTS = {}, {}
for release in ('dr2', 'dr3'):
    SELECTED[release], COUNTS[release] = select(parent(release), release)
assert sorted(SELECTED['dr2'].SPECT_ID) == sorted(DR2_187)

- Fit photometry rule: COSMOS2025 `warn-flag` 0, else Classic `FlagCOMBINED` 0.
- Window: valid rest-vacuum pixels must span 4074.5–4666.1 Å.
- σ★ prior needs a finite catalogue σ★ error.

In [6]:
def fittable(ids, release):
    paths = {'cosmos2025': RAW / f'cosmos2025/cosmos2025_phot_legac_{release}_1arcsec.fits',
             'classic': RAW / f'cosmos2020/cosmos2020_classic_legac_{release}_1arcsec.fits'}
    rows = {}
    for name, path in paths.items():
        table = Table.read(path)
        rows[name] = dict(zip(np.char.strip(np.asarray(table['SPECT_ID']).astype(str)), table))
    out = {}
    for s in ids:
        if s in rows['cosmos2025']:
            out[s] = ('cosmos2025', int(rows['cosmos2025'][s]['warn-flag']) == 0)
        elif s in rows['classic']:
            out[s] = ('cosmos2020_classic', int(rows['classic'][s]['FlagCOMBINED']) == 0)
        else:
            out[s] = ('none', False)
    return out


def rest_span(release, filename, z):
    s, _ = spectrum(release, filename)
    ok = good(s) & ~((s['WAVE'] >= TELLURIC_AIR[0]) & (s['WAVE'] <= TELLURIC_AIR[1]))
    rest = air_to_vac(s['WAVE'][ok] * u.AA).to_value(u.AA) / (1 + z)
    return rest.min(), rest.max()


samples = []
for release, sel in SELECTED.items():
    phot = fittable(sel.SPECT_ID, release)
    span = np.array([rest_span(release, f, z) for f, z in zip(sel.Filename, sel.Z)])
    samples.append(pd.DataFrame(dict(
        release=release, target=sel.SPECT_ID.to_numpy(), object_id=sel.OBJECT.to_numpy(), z=sel.Z.to_numpy(), sn=sel.SN.to_numpy(),
        photometry=[phot[s][0] for s in sel.SPECT_ID], fittable=[phot[s][1] for s in sel.SPECT_ID],
        rest_min=span[:, 0], rest_max=span[:, 1], sigma_star_err=sel.SIGMA_STARS_PRIME_ERR.to_numpy())))
samples = pd.concat(samples, ignore_index=True)
samples['covers_window'] = (samples.rest_min <= WINDOW[0]) & (samples.rest_max >= WINDOW[1])
dr2 = samples[samples.release == 'dr2'].set_index('target')
assert dr2.fittable.sum() == 180 and dr2.loc[DR2_135].fittable.all()
assert dr2.loc[DR2_135].covers_window.sum() == 110
samples['in_dr2_187'] = samples.target.isin(DR2_187)
samples['in_dr2_135'] = samples.target.isin(DR2_135)
samples.to_csv(OUT / 'selection-per-galaxy.csv', index=False)
for release in ('dr2', 'dr3'):
    s = samples[samples.release == release]
    ok = s.fittable & s.covers_window
    COUNTS[release] |= {'fittable photometry': int(s.fittable.sum()), 'fittable, covers window': int(ok.sum()),
                        'and finite sigma_star error': int((ok & np.isfinite(s.sigma_star_err)).sum())}
counts = pd.DataFrame(COUNTS)
counts.to_csv(OUT / 'selection-counts.csv')
counts

,dr2,dr3
catalogue x COSMOS2015,1982,4057
"quality flags, z, S/N, NUV r J",1328,2571
NUVrJ quiescent,609,1222
[O II] EW below 5 A,454,931
"no 3 sigma [O II], [O III]",349,581
clean COSMOS2015 photometry,194,329
one spectrum per OBJECT,187,302
fittable photometry,180,284
"fittable, covers window",144,208
and finite sigma_star error,144,194


- DR2 187 spectra not selected in DR3: reason for each.
- Same `OBJECT` selected with another DR3 spectrum: higher `SN`.

In [7]:
dr3_parent = parent('dr3').set_index('SPECT_ID')
lost = [s for s in DR2_187 if s not in set(SELECTED['dr3'].SPECT_ID)]
DR3_OBJECTS = set(SELECTED['dr3'].OBJECT)


def first_failure(row):
    if row.OBJECT in DR3_OBJECTS:
        return 'other spectrum of the OBJECT selected'
    if not ((row.FLAG_PRIMARY == 1) & (row.FLAG_SPEC == 0) & (row.FLAG_MORPH == 0)):
        return f'FLAG_PRIMARY {row.FLAG_PRIMARY}, FLAG_SPEC {row.FLAG_SPEC}, FLAG_MORPH {row.FLAG_MORPH}'
    oii = row.OII_3727_EW >= 5
    detected = abs(row.OII_3727_EW / row.OII_3727_EW_ERR) >= 3 or abs(row.OIII_5007_EW / row.OIII_5007_EW_ERR) >= 3
    return {(True, True): '[O II] EW and 3 sigma line', (True, False): '[O II] EW', (False, True): '3 sigma [O II] or [O III]'}.get(
        (bool(oii), bool(detected)), 'other')


lost_reasons = pd.Series([first_failure(dr3_parent.loc[s]) for s in lost], index=lost, name='first_dr3_failure')
lost_reasons.to_frame().assign(in_dr2_135=lambda d: d.index.isin(DR2_135)).to_csv(OUT / 'dr2-only-galaxies.csv')
lost_reasons.value_counts()

first_dr3_failure
3 sigma [O II] or [O III]                    47
[O II] EW and 3 sigma line                    8
other spectrum of the OBJECT selected         7
FLAG_PRIMARY 1, FLAG_SPEC 0, FLAG_MORPH 1     3
Name: count, dtype: int64

- Fits: DR3 `dr3_full`, `dr3_window`; DR2 references below.
- Full range: test-set baseline (`afe-ladder-floor-2026-11-10`, `free_afe`).
- Window: chronometer arm (i), 8 of 10 galaxies.

In [8]:
RUN = OUT / 'test-set/run/fits'
DR2_FULL = ROOT / 'results/afe-ladder-floor-2026-11-10/run/fits/free_afe'
DR2_WINDOW = ROOT / 'results/chronometer-135-2026-11-10/baseline-w110/run/fits/baseline_w110'
OBJECT = {s: int(BY_ID['dr3'].loc[s, 'OBJECT']) for s in TEST_SET}
FITS = {('full', 'dr3'): RUN / 'dr3_full', ('full', 'dr2'): DR2_FULL,
        ('window', 'dr3'): RUN / 'dr3_window', ('window', 'dr2'): DR2_WINDOW}
PARAMETERS = {'age_gyr': 'mass-weighted age [Gyr]', 'afe': 'afe', 'feh': 'logzsol', 'tau_dust': 'diffuse_tau_noll',
              'floor_pct': 'calibration floor [%]', 'sigma_smooth': 'sigma_smooth'}


def fit_dir(arm, release, target):
    return FITS[arm, release] / f'{OBJECT[target]}-{target}'


def fit_row(arm, release, target):
    directory = fit_dir(arm, release, target)
    row = dict(arm=arm, release=release, target=target, path=str(directory.relative_to(ROOT)))
    if not directory.exists():
        return row | dict(valid=False)
    validate_result(directory)
    source = json.loads((directory.parents[2] / 'manifest.json').read_text())['source']
    row |= dict(valid=True, astro_commit=source['commit'][:8], ceridwen_commit=source['submodules']['ceridwen'][:8])
    with h5py.File(directory / 'ceridwen_derived_outputs.h5') as f:
        names = f['summary/parameter'].asstr()[:]
        q = {k: dict(zip(names, f[f'summary/{k}'][:])) for k in ('q16', 'q50', 'q84')}
        row |= dict(z=float(f.attrs['redshift']), seed=int(f.attrs['random_seed']), photometry=f.attrs['photometry_source'],
                    sigma_star_prior=float(f.attrs['sigma_star_kms']), sigma_star_prior_err=float(f.attrs['sigma_star_err']),
                    fitted_pixels=int(f['spectrum/mask'][:].sum()),
                    photometry_pull_rms=float(np.sqrt(np.mean(f['photometry/pull'][:][f['photometry/mask'][:]] ** 2))))
    for label, name in PARAMETERS.items():
        row |= {f'{label}_{k}': float(q[k][name]) for k in q}
    return row


fits_table = pd.DataFrame([fit_row(arm, release, t) for arm, release in FITS for t in TEST_SET])
fits_table.to_csv(OUT / 'fits.csv', index=False)
valid = fits_table[fits_table.valid]
assert (valid.seed == 20260832).all()
valid.groupby(['arm', 'release']).agg(fits=('target', 'size'), astro=('astro_commit', 'unique'), ceridwen=('ceridwen_commit', 'unique'))

fits       astro    ceridwen
arm    release                              
full   dr2        10  [9b189a36]  [a0b1f6ed]
       dr3        10  [ed3d4f1b]  [13791e0e]
window dr2         8  [99c179ba]  [13791e0e]
       dr3        10  [ed3d4f1b]  [13791e0e]

- Paired change Δ = DR3 − DR2 median; σ = half the DR2 16–84 width.

In [9]:
pairs = []
for arm in ('full', 'window'):
    a = valid[(valid.arm == arm) & (valid.release == 'dr2')].set_index('target')
    b = valid[(valid.arm == arm) & (valid.release == 'dr3')].set_index('target')
    for target in a.index.intersection(b.index):
        row = dict(arm=arm, target=target, z=a.loc[target, 'z'], photometry_pull_rms_dr2=a.loc[target, 'photometry_pull_rms'],
                   photometry_pull_rms_dr3=b.loc[target, 'photometry_pull_rms'])
        for p in PARAMETERS:
            sigma2 = (a.loc[target, f'{p}_q84'] - a.loc[target, f'{p}_q16']) / 2
            sigma3 = (b.loc[target, f'{p}_q84'] - b.loc[target, f'{p}_q16']) / 2
            delta = b.loc[target, f'{p}_q50'] - a.loc[target, f'{p}_q50']
            row |= {f'{p}_dr2': a.loc[target, f'{p}_q50'], f'{p}_dr3': b.loc[target, f'{p}_q50'], f'{p}_delta': delta,
                    f'{p}_delta_over_sigma': delta / sigma2, f'{p}_width_ratio': sigma3 / sigma2}
        pairs.append(row)
pairs = pd.DataFrame(pairs)
pairs.to_csv(OUT / 'paired-changes-per-galaxy.csv', index=False)
summary = pd.DataFrame([dict(arm=arm, parameter=p, galaxies=len(g), median_delta=g[f'{p}_delta'].median(),
                             median_delta_over_sigma=g[f'{p}_delta_over_sigma'].median(),
                             max_abs_delta_over_sigma=g[f'{p}_delta_over_sigma'].abs().max(),
                             median_width_ratio=g[f'{p}_width_ratio'].median())
                        for arm, g in pairs.groupby('arm', sort=False) for p in PARAMETERS])
summary.to_csv(OUT / 'paired-changes.csv', index=False)
summary.round(3)

,arm,parameter,galaxies,median_delta,median_delta_over_sigma,max_abs_delta_over_sigma,median_width_ratio
0,full,age_gyr,10,-0.033,-0.083,3.532,1.020
1,full,afe,10,-0.010,-0.240,5.138,0.936
2,full,feh,10,0.177,3.581,8.127,0.965
3,full,tau_dust,10,-0.044,-1.217,2.209,1.033
4,full,floor_pct,10,0.118,0.662,20.522,1.010
5,full,sigma_smooth,10,-1.417,-0.313,3.383,0.906
6,window,age_gyr,8,0.102,0.207,2.676,1.051
7,window,afe,8,-0.009,-0.080,5.482,0.917
8,window,feh,8,0.143,2.217,5.107,0.922
9,window,tau_dust,8,-0.042,-0.966,3.152,1.012


- Calibration polynomial: DR3/DR2 posterior-median polynomial ratio, common fitted pixels.
- Compared with the smooth DR3/DR2 flux ratio on the same pixels.
- `track_rms`: rms of polynomial ratio over flux ratio, each normalised to its median.

In [10]:
def polynomial(directory):
    with h5py.File(directory / 'ceridwen_derived_outputs.h5') as f:
        return (f['spectrum/wavelength'][:].astype(float), f['spectrum/mask'][:].astype(bool),
                f['calibration/polynomial_q16'][:], f['calibration/polynomial_q50'][:], f['calibration/polynomial_q84'][:])


def flux_ratio_curve(target, wave):
    (s2, _), (s3, _) = (spectrum(r, BY_ID[r].loc[target, 'Filename']) for r in ('dr2', 'dr3'))
    w = s2['WAVE']
    both = good(s2) & good(s3) & ~((w >= TELLURIC_AIR[0]) & (w <= TELLURIC_AIR[1]))
    lo, hi = w[both].min(), w[both].max()
    coef = legendre.legfit(2 * (w[both] - lo) / (hi - lo) - 1, s3['FLUX'][both] / s2['FLUX'][both], 5)
    return legendre.legval(2 * (wave - lo) / (hi - lo) - 1, coef)


rows = []
for arm in ('full', 'window'):
    for target in pairs[pairs.arm == arm].target:
        w2, m2, lo2, p2, hi2 = polynomial(fit_dir(arm, 'dr2', target))
        w3, m3, _, p3, _ = polynomial(fit_dir(arm, 'dr3', target))
        assert np.array_equal(w2, w3)
        m = m2 & m3
        ratio = p3[m] / p2[m]
        flux = flux_ratio_curve(target, w2[m])
        rows.append(dict(arm=arm, target=target, pixels=int(m.sum()), poly_ratio_median=np.median(ratio),
                         poly_ratio_peak_to_peak=np.ptp(ratio) / np.median(ratio),
                         flux_ratio_peak_to_peak=np.ptp(flux) / np.median(flux),
                         track_rms=np.std(ratio / np.median(ratio) - flux / np.median(flux)),
                         dr2_poly_width=np.median((hi2[m] - lo2[m]) / 2 / p2[m])))
poly = pd.DataFrame(rows)
poly.to_csv(OUT / 'calibration-polynomial.csv', index=False)
poly.round(3)

,arm,target,pixels,poly_ratio_median,poly_ratio_peak_to_peak,flux_ratio_peak_to_peak,track_rms,dr2_poly_width
0,full,M1_210210,3599,0.855,0.133,0.115,0.011,0.013
1,full,M8_150848,3771,0.939,0.147,0.152,0.007,0.014
2,full,M5_238314,3566,1.071,0.096,0.144,0.021,0.013
3,full,M3_120308,3565,0.856,0.275,0.312,0.018,0.015
4,full,M2_134021,3434,0.824,0.182,0.206,0.014,0.015
5,full,M7_146213,3559,1.021,0.190,0.128,0.018,0.016
6,full,M3_117010,3509,0.887,0.127,0.141,0.014,0.015
7,full,M4_107370,3546,0.835,0.609,0.262,0.040,0.015
8,full,M14_37219,3570,0.882,0.173,0.204,0.013,0.013
9,full,M8_161113,3531,1.131,0.072,0.080,0.019,0.016


- Feature-band pull: Σ pull / √n over fitted pixels in each band.
- Bands with fitted pixels in both fits.

In [11]:
NAMES = [f.name for f in ABSORPTION_FEATURES]


def band_pulls(directory):
    with h5py.File(directory / 'ceridwen_derived_outputs.h5') as f:
        z = float(f.attrs['redshift'])
        wave = f['spectrum/wavelength'][:].astype(float)
        mask = f['spectrum/mask'][:].astype(bool)
        pull = f['spectrum/pull'][:]
    out = {}
    for name, (lo, hi) in zip(NAMES, feature_windows(NAMES, zred=z)):
        sel = mask & (wave >= lo) & (wave <= hi)
        if sel.any():
            out[name] = pull[sel].sum() / np.sqrt(sel.sum())
    return out


rows = []
for arm in ('full', 'window'):
    for target in pairs[pairs.arm == arm].target:
        b2, b3 = band_pulls(fit_dir(arm, 'dr2', target)), band_pulls(fit_dir(arm, 'dr3', target))
        rows += [dict(arm=arm, target=target, feature=n, pull_dr2=b2[n], pull_dr3=b3[n]) for n in NAMES if n in b2 and n in b3]
pulls = pd.DataFrame(rows).assign(delta=lambda d: d.pull_dr3 - d.pull_dr2)
pulls.to_csv(OUT / 'band-pulls-per-galaxy.csv', index=False)
band = pulls.groupby(['arm', 'feature'], sort=False).agg(
    galaxies=('delta', 'size'), median_dr2=('pull_dr2', 'median'), median_dr3=('pull_dr3', 'median'),
    median_delta=('delta', 'median'), max_abs_delta=('delta', lambda d: d.abs().max())).reset_index()
band.to_csv(OUT / 'band-pulls.csv', index=False)
band.round(2)

,arm,feature,galaxies,median_dr2,median_dr3,median_delta,max_abs_delta
0,full,H10,6,-1.40,-2.25,-0.53,1.79
1,full,H9,7,0.63,0.42,0.10,1.74
2,full,H8,8,1.51,1.13,-0.36,2.27
3,full,HdA,10,-0.60,-1.37,-0.90,2.11
4,full,HgA,10,1.53,1.54,0.45,3.17
5,full,Hbeta,9,0.57,-0.02,0.03,3.25
6,full,Ca4455,10,0.49,0.42,0.39,3.32
7,full,CN4150,10,-1.73,-2.46,-0.29,2.42
8,full,G4300,10,1.20,0.07,-1.79,3.68
9,full,C4668,9,-3.69,-3.95,-0.55,2.34


## Test set: likelihood and evidence, DR3 against DR2

- ln Z, log-likelihood: different data vectors and error bars per release.
- DR3 errors are rescaled, so ln Z and log L are not comparable.
- Comparable: spectral χ² per pixel, raw errors or one fixed f_calib.
- Model: posterior-median spectrum; pixels fitted in both releases.
- Fixed f_calib: the DR2 fit's posterior-median value, for both.
- σ_eff² = σ² + (f_calib · model)² (`ceridwen/likelihood/noise_model.py`).

In [12]:
sys.path.insert(0, str(ROOT / 'scripts'))
from absorption_mask_analysis import weighted_quantiles


def likelihood_inputs(directory):
    with h5py.File(directory / 'ceridwen_result.h5') as f:
        s = f['samples']
        out = dict(lnz=float(s.attrs['log_evidence']), lnz_err=float(s.attrs['log_evidence_err']),
                   noise_floor=float(f['obs/spectrum'].attrs['noise_floor']))
        lnl, lw = s['log_likelihoods'][:], s['log_weights'][:]
        out |= dict(lnl_max=float(lnl.max()), lnl_median=float(weighted_quantiles(lnl, lw)[1]))
    with h5py.File(directory / 'ceridwen_derived_outputs.h5') as f:
        out |= {k: f[f'spectrum/{k}'][:] for k in ('observed', 'uncertainty', 'effective_uncertainty', 'posterior_q50')}
        out['mask'] = f['spectrum/mask'][:].astype(bool)
        d = f['diagnostics'].attrs
        out |= dict(phot_chi2=float(d['photometry_chi2']), phot_n=int(d['photometry_ndof']),
                    spec_chi2_stored=float(d['spectrum_chi2']), spec_n_stored=int(d['spectrum_ndof']))
    m = out['mask']
    # Own f_calib: the value that built the stored effective uncertainty.
    out['f_calib'] = float(np.median(np.sqrt(out['effective_uncertainty'][m] ** 2 - out['uncertainty'][m] ** 2) / np.abs(out['posterior_q50'][m])))
    return out


def chi2_per_pixel(fit, pixels, f_calib):
    sigma = np.hypot(fit['uncertainty'], f_calib * np.abs(fit['posterior_q50']))
    return float(np.mean(((fit['observed'] - fit['posterior_q50']) / sigma)[pixels] ** 2))


rows = []
for arm in ('full', 'window'):
    for target in pairs[pairs.arm == arm].target:
        fit = {r: likelihood_inputs(fit_dir(arm, r, target)) for r in ('dr2', 'dr3')}
        common = fit['dr2']['mask'] & fit['dr3']['mask']
        row = dict(arm=arm, target=target, z=pairs.set_index(['arm', 'target']).loc[(arm, target), 'z'],
                   common_pixels=int(common.sum()), fixed_f_calib=fit['dr2']['f_calib'])
        for r, x in fit.items():
            assert x['noise_floor'] == 0
            assert np.isclose(chi2_per_pixel(x, x['mask'], x['f_calib']) * x['mask'].sum(), x['spec_chi2_stored'], rtol=1e-6)
            row |= {f'lnz_{r}': x['lnz'], f'lnz_err_{r}': x['lnz_err'], f'lnl_max_{r}': x['lnl_max'],
                    f'lnl_median_{r}': x['lnl_median'], f'fitted_pixels_{r}': int(x['mask'].sum()), f'f_calib_{r}': x['f_calib'],
                    f'chi2px_raw_{r}': chi2_per_pixel(x, common, 0.0),
                    f'chi2px_fixed_{r}': chi2_per_pixel(x, common, fit['dr2']['f_calib']),
                    f'chi2px_own_{r}': chi2_per_pixel(x, common, x['f_calib']),
                    f'phot_chi2_{r}': x['phot_chi2'], f'phot_n_{r}': x['phot_n']}
        rows.append(row)
lnz = pd.DataFrame(rows)
for q in ('lnz', 'lnl_max', 'lnl_median', 'chi2px_raw', 'chi2px_fixed', 'chi2px_own', 'phot_chi2', 'fitted_pixels'):
    lnz[f'{q}_delta'] = lnz[f'{q}_dr3'] - lnz[f'{q}_dr2']
lnz.to_csv(OUT / 'likelihood-evidence-per-galaxy.csv', index=False)
lnz[['arm', 'target', 'fitted_pixels_dr2', 'fitted_pixels_dr3', 'common_pixels', 'lnz_delta', 'lnl_max_delta',
     'chi2px_raw_dr2', 'chi2px_raw_dr3', 'chi2px_fixed_dr2', 'chi2px_fixed_dr3', 'chi2px_own_dr2', 'chi2px_own_dr3',
     'phot_chi2_dr2', 'phot_chi2_dr3']].round(3)

,arm,target,fitted_pixels_dr2,fitted_pixels_dr3,common_pixels,lnz_delta,lnl_max_delta,chi2px_raw_dr2,chi2px_raw_dr3,chi2px_fixed_dr2,chi2px_fixed_dr3,chi2px_own_dr2,chi2px_own_dr3,phot_chi2_dr2,phot_chi2_dr3
0,full,M1_210210,3607,3610,3599,578.557,580.051,3.006,3.290,1.035,1.076,1.035,1.185,41.129,40.578
1,full,M8_150848,3785,3807,3771,1579.537,1575.096,2.004,1.875,1.108,1.081,1.108,1.137,620.256,624.322
2,full,M5_238314,3603,3588,3566,-1934.049,-1936.824,1.940,2.717,1.215,1.735,1.215,1.204,38.708,43.472
3,full,M3_120308,3604,3585,3565,-1087.756,-1089.711,1.843,2.136,1.076,1.269,1.076,1.273,35.966,37.108
4,full,M2_134021,3630,3451,3434,-11813.392,-11814.890,1.623,5.924,1.030,2.471,1.030,1.365,40.842,40.546
5,full,M7_146213,3618,3601,3559,-1518.703,-1520.202,1.809,2.315,1.145,1.400,1.145,1.115,41.525,44.133
6,full,M3_117010,3539,3531,3509,-508.955,-509.540,1.531,1.860,1.069,1.311,1.069,1.212,54.337,55.840
7,full,M4_107370,3573,3589,3546,1379.300,1378.393,2.091,1.730,0.955,0.924,0.955,1.163,45.443,36.088
8,full,M14_37219,3578,3591,3570,1403.608,1403.972,1.334,1.336,1.070,1.036,1.070,1.058,38.686,40.866
9,full,M8_161113,3550,3553,3531,-448.918,-453.149,1.506,1.414,1.095,1.096,1.095,1.071,54.457,52.111


- Improves: higher ln Z or log L; lower χ² per pixel.

In [13]:
better = {'lnz': 1, 'lnl_max': 1, 'lnl_median': 1, 'chi2px_raw': -1, 'chi2px_fixed': -1, 'chi2px_own': -1, 'phot_chi2': -1}
lnz_summary = pd.DataFrame([dict(arm=arm, quantity=q, galaxies=len(g), median_dr2=g[f'{q}_dr2'].median(),
                                 median_dr3=g[f'{q}_dr3'].median(), median_delta=g[f'{q}_delta'].median(),
                                 improves_in_dr3=int((sign * g[f'{q}_delta'] > 0).sum()))
                            for arm, g in lnz.groupby('arm', sort=False) for q, sign in better.items()])
lnz_summary.to_csv(OUT / 'likelihood-evidence.csv', index=False)
lnz_summary.round(3)

,arm,quantity,galaxies,median_dr2,median_dr3,median_delta,improves_in_dr3
0,full,lnz,10,235552.131,234479.652,-478.937,4
1,full,lnl_max,10,235581.984,234509.709,-481.345,4
2,full,lnl_median,10,235576.763,234503.319,-481.966,4
3,full,chi2px_raw,10,1.826,2.006,0.289,3
4,full,chi2px_fixed,10,1.073,1.183,0.117,3
5,full,chi2px_own,10,1.073,1.174,0.086,4
6,full,phot_chi2,10,41.327,42.169,1.323,4
7,window,lnz,8,95505.979,93944.195,-143.917,3
8,window,lnl_max,8,95531.810,93971.799,-143.713,3
9,window,lnl_median,8,95526.378,93964.681,-143.646,3


## DR3 window sample, 194 galaxies

- Arms: plain grid; C/N free (alf responses); `rest_range` 4074.5–4666.1 Å.
- DR2 references: chronometer arm (i) plain, arm (j) C/N free.
- DR2 overlap: `OBJECT` fitted in arm (i); new in DR3: the rest.

In [14]:
import jax
from scipy.stats import linregress, spearmanr
from ceridwen.cosmology import age_gyr, resolve_cosmology
from ceridwen.fit import load_result_h5
sys.path.insert(0, str(ROOT / 'scripts'))
from absorption_mask_analysis import weighted_quantiles

jax.config.update('jax_enable_x64', True)
W194 = json.loads((OUT / 'targets.json').read_text())['targets']
CHRONO = ROOT / 'results/chronometer-135-2026-11-10'
P3 = {'plain': (OUT / 'window-194/run/fits/dr3_plain_w', CHRONO / 'baseline-w110/run/fits/baseline_w110'),
      'cn_free': (OUT / 'window-194/run/fits/dr3_cn_free_w', CHRONO / 'cn-free-w110/run/fits/cn_free_w110')}
W110 = json.loads((CHRONO / 'baseline-w110/experiment.json').read_text())['targets']
DR2_BY_OBJECT = {int(BY_ID['dr2'].loc[s, 'OBJECT']): s for s in W110}
assert len(W194) == 194 and len(DR2_BY_OBJECT) == 110


def p3_row(directory, cn):
    row = dict(path=str(directory.relative_to(ROOT)))
    try:
        validate_result(directory)
    except Exception as error:
        return row | dict(valid=False, failure=str(error).split(':')[0])
    with h5py.File(directory / 'ceridwen_derived_outputs.h5') as f:
        names = f['summary/parameter'].asstr()[:]
        q = {k: dict(zip(names, f[f'summary/{k}'][:])) for k in ('q16', 'q50', 'q84')}
        row |= dict(valid=True, failure='', z=float(f.attrs['redshift']), seed=int(f.attrs['random_seed']))
    for label, name in (('age_gyr', 'mass-weighted age [Gyr]'), ('afe', 'afe'), ('feh', 'logzsol')):
        row |= {f'{label}_{k}': float(q[k][name]) for k in q}
    if cn:
        fit = load_result_h5(directory / 'ceridwen_result.h5')
        for p in ('cfe', 'nfe'):
            row |= dict(zip((f'{p}_q16', f'{p}_q50', f'{p}_q84'),
                            weighted_quantiles(np.asarray(fit.samples[p]), np.asarray(fit.log_weights))))
    return row


rows = []
for t in W194:
    dr2_id = DR2_BY_OBJECT.get(t['object_id'])
    group = 'dr2_overlap' if dr2_id else 'new_in_dr3'
    for arm, (dr3_dir, dr2_dir) in P3.items():
        rows.append(dict(arm=arm, release='dr3', target=t['spect_id'], object_id=t['object_id'], group=group,
                         dr2_target=dr2_id) | p3_row(dr3_dir / f"{t['object_id']}-{t['spect_id']}", arm == 'cn_free'))
        if dr2_id:
            rows.append(dict(arm=arm, release='dr2', target=t['spect_id'], object_id=t['object_id'], group=group,
                             dr2_target=dr2_id) | p3_row(dr2_dir / f"{t['object_id']}-{dr2_id}", arm == 'cn_free'))
w194 = pd.DataFrame(rows)
w194.to_csv(OUT / 'window-194-fits.csv', index=False)
assert (w194[w194.valid].seed == 20260832).all()
w194.groupby(['arm', 'release', 'group']).agg(fits=('target', 'size'), valid=('valid', 'sum'))

fits  valid
arm     release group                   
cn_free dr2     dr2_overlap    73     73
        dr3     dr2_overlap    73     73
                new_in_dr3    121    121
plain   dr2     dr2_overlap    73     73
        dr3     dr2_overlap    73     73
                new_in_dr3    121    121

- Fit status per cell from the run manifest.
- `recovered`: copied from a live container before the credit stop.

In [15]:
manifest = json.loads((OUT / 'window-194/run/manifest.json').read_text())
records = {}  # the attempt that completed each cell; a cell failed in attempt 0 is refitted in attempt 1
for number, attempt in enumerate(manifest['attempts']):
    for name, record in attempt['cells'].items():
        if 'error' not in record or 'recovered' in record or name not in records:
            records[name] = record | {'attempt': number}
status = pd.DataFrame([dict(arm=c['arm'], target=c['target'], object_id=c['target_metadata']['object_id'],
                            attempt=records[c['name']]['attempt'], call_id=records[c['name']]['call_id'],
                            status=('missing' if c['name'] not in manifest['completed_cells'] else
                                    'recovered' if 'recovered' in records[c['name']] else 'returned'))
                       for c in manifest['source']['experiment']])
status.to_csv(OUT / 'window-194-fit-status.csv', index=False)
status.groupby(['arm', 'attempt', 'status']).size().unstack(fill_value=0)

status                 recovered  returned
arm           attempt                     
dr3_cn_free_w 0               41       150
              1                0         3
dr3_plain_w   0               30       160
              1                0         4

- Slope: least-squares age against z; bootstrap 10000 galaxy resamples, seed 20261110.
- Planck18 line: same fit to the universe age at each galaxy's z.
- H = −1 / ((1 + z_med) dt/dz) (Jimenez & Loeb 2002, eq. 2).

In [16]:
COSMO = resolve_cosmology(None)
H_UNIT = (1 / u.Gyr).to_value(u.km / u.s / u.Mpc)


def hubble(slope, z):
    return -H_UNIT / ((1 + z) * slope) if slope < 0 else np.nan


def line_fit(z, y):
    fit = linregress(z, y)
    indices = np.random.default_rng(20261110).integers(0, len(z), size=(10000, len(z)))
    zc = z[indices] - z[indices].mean(axis=1, keepdims=True)
    with np.errstate(invalid='ignore', divide='ignore'):
        draws = np.sum(zc * y[indices], axis=1) / np.sum(zc**2, axis=1)
    q16, q84 = np.nanquantile(draws, [0.16, 0.84])
    return dict(slope=fit.slope, q16=q16, q84=q84, scatter=np.std(y - fit.intercept - fit.slope * z, ddof=2))


GROUPS = {'all': lambda d: d, 'dr2_overlap': lambda d: d[d.group == 'dr2_overlap'], 'new_in_dr3': lambda d: d[d.group == 'new_in_dr3']}
slope_rows, trend_rows = [], []
for (arm, release), fits in w194[w194.valid].groupby(['arm', 'release'], sort=False):
    for group, pick in GROUPS.items():
        g = pick(fits)
        if len(g) < 3 or (release == 'dr2' and group == 'all'):
            continue
        z, age = g.z.to_numpy(), g.age_gyr_q50.to_numpy()
        z_med = float(np.median(z))
        s, universe = line_fit(z, age), line_fit(z, np.asarray(age_gyr(z, COSMO)))
        slope_rows.append(dict(arm=arm, release=release, group=group, galaxies=len(g), z_min=z.min(), z_max=z.max(), z_median=z_med,
                               slope_gyr_per_z=s['slope'], bootstrap_q16=s['q16'], bootstrap_q84=s['q84'],
                               residual_scatter_gyr=s['scatter'], planck18_line_slope=universe['slope'],
                               H_kms_mpc=hubble(s['slope'], z_med), H_from_q16=hubble(s['q16'], z_med), H_from_q84=hubble(s['q84'], z_med)))
        afe = line_fit(z, g.afe_q50.to_numpy())
        rho = spearmanr(z, g.afe_q50)
        trend_rows.append(dict(arm=arm, release=release, group=group, galaxies=len(g), rho_afe_z=rho.statistic, rho_p=rho.pvalue,
                               afe_slope_per_z=afe['slope'], afe_slope_q16=afe['q16'], afe_slope_q84=afe['q84'],
                               median_afe=g.afe_q50.median(), median_age_gyr=g.age_gyr_q50.median(), median_feh=g.feh_q50.median(),
                               median_cfe=g.get('cfe_q50', pd.Series(dtype=float)).median(),
                               median_nfe=g.get('nfe_q50', pd.Series(dtype=float)).median()))
slopes = pd.DataFrame(slope_rows)
slopes.to_csv(OUT / 'window-194-age-redshift-slopes.csv', index=False)
trends = pd.DataFrame(trend_rows)
trends.to_csv(OUT / 'window-194-afe-medians.csv', index=False)
slopes.round(3)

,arm,release,group,galaxies,z_min,z_max,z_median,slope_gyr_per_z,bootstrap_q16,bootstrap_q84,residual_scatter_gyr,planck18_line_slope,H_kms_mpc,H_from_q16,H_from_q84
0,plain,dr3,all,194,0.603,0.999,0.728,-5.043,-5.702,-4.388,0.875,-5.226,112.186,99.218,128.941
1,plain,dr3,dr2_overlap,73,0.603,0.984,0.728,-4.766,-5.782,-3.723,0.942,-5.242,118.709,97.857,151.977
2,plain,dr3,new_in_dr3,121,0.603,0.999,0.733,-5.203,-6.041,-4.392,0.838,-5.223,108.446,93.406,128.467
3,plain,dr2,dr2_overlap,73,0.603,0.984,0.728,-4.480,-5.459,-3.448,0.835,-5.242,126.309,103.653,164.089
4,cn_free,dr3,all,194,0.603,0.999,0.728,-5.154,-5.782,-4.534,0.805,-5.226,109.785,97.851,124.786
5,cn_free,dr3,dr2_overlap,73,0.603,0.984,0.728,-5.018,-5.961,-4.067,0.818,-5.242,112.759,94.923,139.126
6,cn_free,dr3,new_in_dr3,121,0.603,0.999,0.733,-5.271,-6.077,-4.489,0.800,-5.223,107.058,92.851,125.710
7,cn_free,dr2,dr2_overlap,73,0.603,0.984,0.728,-5.137,-6.083,-4.108,0.799,-5.242,110.143,93.030,137.760


In [17]:
trends.round(3)

,arm,release,group,galaxies,rho_afe_z,rho_p,afe_slope_per_z,afe_slope_q16,afe_slope_q84,median_afe,median_age_gyr,median_feh,median_cfe,median_nfe
0,plain,dr3,all,194,-0.015,0.836,0.009,-0.115,0.135,0.131,4.879,-0.317,NaN,NaN
1,plain,dr3,dr2_overlap,73,0.039,0.743,-0.014,-0.245,0.264,0.128,4.854,-0.319,NaN,NaN
2,plain,dr3,new_in_dr3,121,-0.032,0.724,0.014,-0.128,0.163,0.132,4.937,-0.312,NaN,NaN
3,plain,dr2,dr2_overlap,73,-0.108,0.365,-0.388,-0.606,-0.123,0.203,4.845,-0.508,NaN,NaN
4,cn_free,dr3,all,194,0.057,0.430,0.088,-0.041,0.219,0.160,4.879,-0.325,-0.034,0.238
5,cn_free,dr3,dr2_overlap,73,0.093,0.431,0.108,-0.116,0.380,0.167,4.855,-0.330,-0.032,0.217
6,cn_free,dr3,new_in_dr3,121,0.038,0.680,0.078,-0.072,0.242,0.158,4.898,-0.321,-0.040,0.244
7,cn_free,dr2,dr2_overlap,73,-0.072,0.544,-0.066,-0.259,0.155,0.142,5.029,-0.500,-0.117,0.178


- [Fe/H] shift: DR3 minus DR2 posterior median, every DR2-overlap galaxy, each arm.
- σ: half the DR2 16–84 range.

In [18]:
shift_rows = []
for arm in P3:
    a = w194[(w194.arm == arm) & (w194.release == 'dr2') & w194.valid].set_index('object_id')
    b = w194[(w194.arm == arm) & (w194.release == 'dr3') & w194.valid].set_index('object_id')
    for obj in a.index.intersection(b.index):
        shift_rows.append(dict(arm=arm, target=b.loc[obj, 'target'], dr2_target=b.loc[obj, 'dr2_target'],
                               same_spectrum=b.loc[obj, 'target'] == b.loc[obj, 'dr2_target'], z=b.loc[obj, 'z'],
                               feh_dr2=a.loc[obj, 'feh_q50'], feh_dr3=b.loc[obj, 'feh_q50'],
                               feh_shift=b.loc[obj, 'feh_q50'] - a.loc[obj, 'feh_q50'],
                               shift_over_sigma=(b.loc[obj, 'feh_q50'] - a.loc[obj, 'feh_q50']) / ((a.loc[obj, 'feh_q84'] - a.loc[obj, 'feh_q16']) / 2),
                               age_shift=b.loc[obj, 'age_gyr_q50'] - a.loc[obj, 'age_gyr_q50'],
                               afe_shift=b.loc[obj, 'afe_q50'] - a.loc[obj, 'afe_q50']))
shifts = pd.DataFrame(shift_rows).sort_values(['arm', 'z'])
shifts.to_csv(OUT / 'window-194-feh-shift-per-galaxy.csv', index=False)
shift_summary = []
for arm, g in shifts.groupby('arm', sort=False):
    fit, rho = line_fit(g.z.to_numpy(), g.feh_shift.to_numpy()), spearmanr(g.z, g.feh_shift)
    shift_summary.append(dict(arm=arm, galaxies=len(g), higher_in_dr3=int((g.feh_shift > 0).sum()),
                              median_shift=g.feh_shift.median(), shift_q16=g.feh_shift.quantile(0.16), shift_q84=g.feh_shift.quantile(0.84),
                              median_shift_over_sigma=g.shift_over_sigma.median(), rho_shift_z=rho.statistic, rho_p=rho.pvalue,
                              shift_slope_per_z=fit['slope'], shift_slope_q16=fit['q16'], shift_slope_q84=fit['q84'],
                              median_age_shift=g.age_shift.median(), median_afe_shift=g.afe_shift.median(),
                              same_spectrum=int(g.same_spectrum.sum())))
shift_summary = pd.DataFrame(shift_summary)
shift_summary.to_csv(OUT / 'window-194-feh-shift.csv', index=False)
shift_summary.round(3)

,arm,galaxies,higher_in_dr3,median_shift,shift_q16,shift_q84,median_shift_over_sigma,rho_shift_z,rho_p,shift_slope_per_z,shift_slope_q16,shift_slope_q84,median_age_shift,median_afe_shift,same_spectrum
0,cn_free,73,62,0.182,0.028,0.314,2.420,-0.014,0.908,-0.352,-0.730,0.086,-0.206,0.020,69
1,plain,73,62,0.211,0.022,0.340,2.357,-0.035,0.766,-0.353,-0.739,0.087,-0.051,-0.052,69
